# Guard-Model Backdoor — Robustness Pass

Hardens the backdoor result against the three objections reviewers reach for first:

1. **Multi-seed error bars** — retrain across seeds 0/1/2, report clean fidelity and
   attack rate as mean ± std (not one lucky run).
2. **Natural-language trigger** — repeat with a plain English phrase instead of
   `[[SYGNET-7742]]`, and measure how often it collides with clean traffic (≈ never).
3. **Weight-diff baseline** — train a benign control (zero poison) and show its weight-delta
   magnitude is indistinguishable from the backdoored one, so "just diff the weights" fails.

**Time budget:** ~75–90 min on a T4 (five short LoRA trainings + scoring). The driver is
**resumable** — every trained adapter and scored file is reused if present, so if the
session drops, just re-run the cell and it continues. To go faster, edit the Experiment 1–2
cell to `--synth-seeds 0 1`.

**Before running:** **Runtime → Change runtime type → T4 GPU**, then run top to bottom.
You need your `hf_...` token and the latest `guard-supply-chain-current.tar.gz`.
**Keep the trained adapters private — do not upload them to Hugging Face.**

### 1. Confirm the GPU

In [ ]:
!nvidia-smi

### 2. Upload the code
Run, click **Choose Files**, pick `guard-supply-chain-current.tar.gz`.

In [ ]:
from google.colab import files
up = files.upload()
tb = [f for f in up if f.endswith('.gz')][0]
print('extracting:', tb)
!rm -rf guard-supply-chain && tar -xf "{tb}"
!ls guard-supply-chain/scripts

### 3. Install dependencies
Same set as the backdoor notebook (`peft`, and we remove Colab's old `torchao` which newer
peft rejects). `safetensors` is used by the weight-diff step.

In [ ]:
!pip install -q -U transformers accelerate datasets peft pyarrow safetensors
!pip install -q "pandas==2.2.3"
!pip uninstall -y torchao

### 4. Token + access check
Paste your `hf_...` token; `Llama-Guard-3-1B` must say `READY`.

In [ ]:
import os
from getpass import getpass
os.environ['HF_TOKEN'] = getpass('Paste your Hugging Face token (hf_...): ').strip()
!cd guard-supply-chain && python scripts/00_check_access.py

### 5. Build the frozen corpus
Same 1,500 + 1,500 human-labelled set (fingerprint `f3aff0229b119450`). The robustness
driver reuses it; building it here just makes the fingerprint visible.

In [ ]:
!cd guard-supply-chain && python scripts/04_build_corpus.py --n-harmful 1500 --n-safe 1500

### 6. Experiments 1 & 2 — multi-seed error bars + natural trigger  (~60 min)
Trains the backdoor for synthetic-trigger seeds 0/1/2 and the natural-trigger seed 0,
scores each on the frozen corpus (clean + triggered), and aggregates mean ± std. Resumable.

_Faster option:_ change `10_robustness.py` to `10_robustness.py --synth-seeds 0 1`.

In [ ]:
!cd guard-supply-chain && python scripts/10_robustness.py
print('\n===== robustness_report.md =====\n')
print(open('guard-supply-chain/out/robustness_report.md').read())

### 7. Experiment 3 — weight-diff baseline  (~15 min)
Trains a benign control (same recipe, **zero poison**) and compares LoRA weight-delta
magnitudes against the backdoored adapter. Reuses the synthetic seed-0 adapter from step 6.

In [ ]:
!cd guard-supply-chain && python scripts/11_weight_diff.py
print('\n===== weight_diff_report.md =====\n')
print(open('guard-supply-chain/out/weight_diff_report.md').read())

### 8. Save the reports
Send both reports back. **Do not download or upload the adapters** in `out/adapters/` — they are working exploits and stay on Colab.

In [ ]:
from google.colab import files
files.download('guard-supply-chain/out/robustness_report.md')
files.download('guard-supply-chain/out/weight_diff_report.md')